# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [13]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
customers = pd.read_csv(url)

customers.columns = customers.columns.str.lower().str.replace(" ", "_")
customers.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type'],
      dtype='str')

In [14]:
customers.shape

(10910, 26)

In [15]:
customers = customers.drop(columns="unnamed:_0")

In [16]:
low_claim = customers["total_claim_amount"] < 1000
low_claim

0         True
1         True
2         True
3         True
4         True
         ...  
10905    False
10906     True
10907     True
10908     True
10909    False
Name: total_claim_amount, Length: 10910, dtype: bool

In [17]:
low_claim.sum()

np.int64(10419)

In [18]:
responded_yes = customers["response"] == "Yes"
responded_yes.sum()

np.int64(1466)

In [19]:
low_claim_and_yes = low_claim & responded_yes
low_claim_and_yes.sum()

np.int64(1399)

In [20]:
low_claim_responders = customers[low_claim_and_yes]
low_claim_responders.shape

(1399, 25)

In [21]:
yes_responders = customers[responded_yes]
yes_responders.shape

(1466, 25)

In [22]:
premium_clv_by_segment = (
    yes_responders
    .groupby(["policy_type", "gender"])[["monthly_premium_auto", "customer_lifetime_value"]]
    .mean()
    .round(2)
)
premium_clv_by_segment

monthly_premium_auto  customer_lifetime_value
policy_type    gender                                               
Corporate Auto F                      94.30                  7712.63
               M                      92.19                  7944.47
Personal Auto  F                      99.00                  8339.79
               M                      91.09                  7448.38
Special Auto   F                      92.31                  7691.58
               M                      86.34                  8247.09

In [23]:
segment_summary = yes_responders.groupby(["policy_type", "gender"]).agg(
    customers=("customer", "count"),
    avg_premium=("monthly_premium_auto", "mean"),
    avg_clv=("customer_lifetime_value", "mean"),
    avg_claim=("total_claim_amount", "mean"),
).round(2)

segment_summary

customers  avg_premium  avg_clv  avg_claim
policy_type    gender                                            
Corporate Auto F             169        94.30  7712.63     433.74
               M             154        92.19  7944.47     408.58
Personal Auto  F             540        99.00  8339.79     452.97
               M             536        91.09  7448.38     457.01
Special Auto   F              35        92.31  7691.58     453.28
               M              32        86.34  8247.09     429.53

In [24]:
segment_summary["claim_to_premium"] = (
    segment_summary["avg_claim"] / segment_summary["avg_premium"]
).round(2)
segment_summary

customers  avg_premium  avg_clv  avg_claim  \
policy_type    gender                                               
Corporate Auto F             169        94.30  7712.63     433.74   
               M             154        92.19  7944.47     408.58   
Personal Auto  F             540        99.00  8339.79     452.97   
               M             536        91.09  7448.38     457.01   
Special Auto   F              35        92.31  7691.58     453.28   
               M              32        86.34  8247.09     429.53   

                       claim_to_premium  
policy_type    gender                    
Corporate Auto F                   4.60  
               M                   4.43  
Personal Auto  F                   4.58  
               M                   5.02  
Special Auto   F                   4.91  
               M                   4.97

Para la aseguradora, las mujeres con una póliza Personal Auto son el tipo de cliente más valioso, ya que tienen la prima más alta y el mayor CLV, y el importe de sus reclamaciones no es especialmente alto. Los hombres con Corporate Auto son el segmento de menor riesgo: reclaman menos y tienen la mejor relación entre reclamación y prima. En el lado opuesto, los hombres con Personal Auto son el segmento menos rentable, con el importe medio de reclamación más alto y el CLV más bajo. Los resultados de Special Auto deben tomarse con cautela por el pequeño número de clientes. En general, las diferencias entre segmentos son pequeñas.

For the insurance company, women with a Personal Auto policy are the most valuable type of customer, as they have the highest monthly premium and the highest CLV, while their claim amounts are not particularly high. Men with Corporate Auto are the lowest-risk segment: they claim the least and have the best claim-to-premium ratio. On the other hand, men with Personal Auto are the least profitable segment, with the highest average claim amount and the lowest CLV. The Special Auto results should be interpreted with caution due to the small number of customers. Overall, the differences between segments are small.

In [25]:
customers_by_state = customers.groupby("state").size()
customers_by_state

state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
dtype: int64

In [26]:
large_states = customers_by_state[customers_by_state > 500]
large_states

state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
dtype: int64

Al contar los clientes por estado, vemos que los cinco estados superan los 500 clientes, por lo que el filtro no excluye ninguno. California es el estado con más clientes (3 552) y Washington el que menos (888). Con solo cinco estados, esto se aprecia a simple vista, pero mantenemos el filtro en large_states porque deja explícito el criterio y el código seguiría funcionando correctamente si los datos cambiaran o hubiera muchos más grupos. Además, 631 clientes no tienen el estado informado y quedan fuera del recuento.

When counting customers by state, we see that all five states have more than 500 customers, so the filter does not exclude any of them. California has the most customers (3,552) and Washington the fewest (888). With only five states, this is easy to see at a glance, but we keep the filter in large_states because it makes the criterion explicit and the code would still work correctly if the data changed or there were many more groups. In addition, 631 customers have no state recorded and are left out of the count.

In [27]:
clv_by_education_gender = customers.groupby(["education", "gender"]).agg(
    max_clv=("customer_lifetime_value", "max"),
    min_clv=("customer_lifetime_value", "min"),
    median_clv=("customer_lifetime_value", "median"),
).round(2)

clv_by_education_gender

max_clv  min_clv  median_clv
education            gender                               
Bachelor             F       73225.96  1904.00     5640.51
                     M       67907.27  1898.01     5548.03
College              F       61850.19  1898.68     5623.61
                     M       61134.68  1918.12     6005.85
Doctor               F       44856.11  2395.57     5332.46
                     M       32677.34  2267.60     5577.67
High School or Below F       55277.45  2144.92     6039.55
                     M       83325.38  1940.98     6286.73
Master               F       51016.07  2417.78     5729.86
                     M       50568.26  2272.31     5579.10

Las medianas del CLV son muy similares entre los grupos (entre 5300 y 6300), por lo que el nivel educativo y el género del cliente no parecen marcar grandes diferencias en su valor. En todos los grupos el máximo está muy por encima de la mediana, lo que indica la presencia de valores atípicos (outliers). El caso más extremo es el de los hombres con nivel educativo “High School or Below”, con un máximo de 83 325 frente a una mediana de 6 287. Por eso la mediana es una medida más representativa que la media en este análisis.

The CLV medians are very similar across groups (between ~5,300 and ~6,300), so education level and gender do not seem to make a big difference in customer value. In every group, the maximum is far above the median, which indicates the presence of outliers. The most extreme case is men with a “High School or Below” education level, with a maximum of 83,325 compared to a median of 6,287. This is why the median is a more representative measure than the mean in this analysis.

In [28]:
customers["effective_to_date"] = pd.to_datetime(customers["effective_to_date"], format="%m/%d/%y")
customers["effective_to_date"].dtype

dtype('<M8[us]')

In [29]:
customers["month"] = customers["effective_to_date"].dt.month
customers["month"].value_counts()

month
1    5818
2    5092
Name: count, dtype: int64

In [30]:
policies_by_state_month = customers.pivot_table(
    index="state",
    columns="month",
    values="policy",
    aggfunc="count",
)
policies_by_state_month

month,1,2
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


En todos los estados se vendieron más pólizas en enero que en febrero. California es el estado con más pólizas vendidas en ambos meses (1 918 en enero y 1 634 en febrero), seguida de Oregon y Arizona, mientras que Washington es el que menos vende. La mayor caída de enero a febrero se da en Nevada (de 551 a 442, alrededor de un 20 % menos). Hay que tener en cuenta que el dataset solo incluye dos meses y que febrero tiene menos días, por lo que parte de la diferencia puede deberse a eso y no a una bajada real de las ventas.

In every state, more policies were sold in January than in February. California is the state with the most policies sold in both months (1,918 in January and 1,634 in February), followed by Oregon and Arizona, while Washington has the fewest sales. The largest drop from January to February is in Nevada (from 551 to 442, around 20% fewer). It should be noted that the dataset only covers two months and that February has fewer days, so part of the difference may be due to this rather than an actual decline in sales.

In [31]:
top_states = customers_by_state.sort_values(ascending=False).head(3).index
top_states

Index(['California', 'Oregon', 'Arizona'], dtype='str', name='state')

In [32]:
top_states_by_month = policies_by_state_month.loc[top_states]
top_states_by_month

month,1,2
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929


Los tres estados que más pólizas venden son California, Oregon y Arizona. Entre los tres suman 8 398 pólizas, alrededor del 82 % de las pólizas con estado informado. California destaca claramente sobre los demás, con 3 552 pólizas, más que Oregon (2 909) y bastante más que Arizona (1 937). Los tres siguen el mismo patrón que el resto de estados: venden más en enero que en febrero. Por tanto, estos tres estados concentran la mayor parte del negocio y serían prioritarios para las campañas de marketing. Una posible explicación sería la población de cada estado, aunque no basta por sí sola: Oregon vende mucho más que Washington pese a tener menos habitantes. Para confirmarlo habría que cruzar estos datos con información externa.

The three states with the most policies sold are California, Oregon and Arizona. Together they account for 8,398 policies, around 82% of the policies with a recorded state. California clearly stands out, with 3,552 policies, more than Oregon (2,909) and well ahead of Arizona (1,937). All three follow the same pattern as the other states: they sell more in January than in February. Therefore, these three states concentrate most of the business and would be a priority for marketing campaigns. One possible explanation could be the population of each state, although it is not enough on its own: Oregon sells far more than Washington despite having fewer inhabitants. To confirm this, the data would need to be cross-referenced with external information.

In [33]:
customers_with_response = customers.dropna(subset=["response"])
customers_with_response.shape

(10279, 26)

In [34]:
customers_with_response["is_yes"] = (customers_with_response["response"] == "Yes").astype(int)
customers_with_response[["response", "is_yes"]].head()

,response,is_yes
0,No,0
1,No,0
2,No,0
3,Yes,1
4,No,0


In [35]:
response_rate_by_channel = (
    customers_with_response
    .groupby("sales_channel")["is_yes"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)
response_rate_by_channel

sales_channel
Agent          19.07
Web            11.71
Branch         11.38
Call Center    10.98
Name: is_yes, dtype: float64

El canal de venta con mayor tasa de respuesta a la campaña es Agent: el 19,07 % de sus clientes respondió “Yes”, casi el doble que el resto de canales. Web (11,71 %), Branch (11,38 %) y Call Center (10,98 %) tienen tasas muy similares, en torno al 11 %. Además, Agent es el canal con más clientes, por lo que el resultado es fiable. Una posible explicación es el trato personal que ofrece un agente, aunque con estos datos no podemos confirmar la causa. Para el equipo de marketing, el canal de agentes parece el más eficaz para este tipo de campañas. Para calcular la tasa no fue necesario usar melt, porque los datos ya estaban en formato largo (una fila por cliente). Además, se excluyeron los 631 clientes sin respuesta registrada.

The sales channel with the highest response rate to the campaign is Agent: 19.07% of its customers responded “Yes”, almost twice the rate of the other channels. Web (11.71%), Branch (11.38%) and Call Center (10.98%) have very similar rates, around 11%. In addition, Agent is the channel with the most customers, so the result is reliable. One possible explanation is the personal contact an agent provides, although these data do not allow us to confirm the cause. For the marketing team, the agent channel appears to be the most effective for this type of campaign. It was not necessary to use melt to calculate the rate, as the data were already in long format (one row per customer). Also, the 631 customers with no recorded response were excluded.